New Notebook Created by Jupyter MCP Server

> **Nhận xét — Ô khởi tạo**
>
> Cell auto sinh bởi MCP khi `use_notebook(mode="create")`. Đánh dấu notebook được agent tạo, không viết tay.

# MCP Edge Case Demo

Scenarios where **raw file edit fails** but **MCP wins**:

1. Silent fallback on malformed IDs
2. Subject-leakage assertion
3. Kernel restart wipes state
4. Out-of-order execution recovery

> **Nhận xét — Mục lục demo**
>
> 4 edge case được chọn vì raw file edit **không xử lý được**:
> 1. Hành vi chỉ lộ khi execute
> 2. Assertion runtime trên data thật
> 3. Kernel state / dependency chain
> 4. Bug logic ẩn + surgical fix loop

In [10]:
# EDGE 1: silent fallback — only visible after execution
from pathlib import Path
import sys


def _ensure_scripts_on_path() -> None:
    root = next(
        (p for p in (Path.cwd(), *Path.cwd().parents)
         if (p / "scripts" / "assign_subject_splits.py").is_file()),
        Path("/home/tiencd/rf-worldpose"),
    )
    scripts = str(root / "scripts")
    if scripts not in sys.path:
        sys.path.insert(0, scripts)


_ensure_scripts_on_path()
from assign_subject_splits import _MMFI_RE, _mmfi_split

CASES = (
    ("invalid literal", "INVALID"),
    ("missing action", "E1_S5"),
    ("empty string", ""),
    ("valid -> test", "E1_S5_A1"),
    ("garbage", "garbage_id"),
    ("valid -> test", "E99_S05_A01"),
)

W_REASON, W_ID = 18, 16
lines = [
    f"{'reason':<{W_REASON}} {'sample_id':<{W_ID}} regex  split",
    "-" * 52,
]
silent = 0
for reason, sid in CASES:
    ok = _MMFI_RE.match(sid) is not None
    split = _mmfi_split(sid)
    silent += not ok
    lines.append(f"{reason:<{W_REASON}} {sid!r:<{W_ID}} {str(ok):>5}  {split!r}")

print("Silent fallback demo (unmatched IDs default to 'train'):\n")
print("\n".join(lines))
print(f"\nSummary: {silent}/{len(CASES)} IDs silently routed to train")

Silent fallback demo (unmatched IDs default to 'train'):

reason             sample_id        regex  split
----------------------------------------------------
invalid literal    'INVALID'        False  'train'
missing action     'E1_S5'          False  'train'
empty string       ''               False  'train'
valid -> test      'E1_S5_A1'        True  'test'
garbage            'garbage_id'     False  'train'
valid -> test      'E99_S05_A01'     True  'test'

Summary: 4/6 IDs silently routed to train


> **Nhận xét EDGE 1** (MCP `overwrite_cell_source` + `execute_cell`)
>
> - `_ensure_scripts_on_path()` tự walk repo root — không hardcode path.
> - Một vòng lặp: build `lines` + đếm `silent` — không list `rows` trung gian.
> - Cột `regex` vs `split` tách rõ ID hợp lệ / fallback im lặng → `'train'`.
> - **4/6 ID** không match regex vẫn vào train — output lấy từ kernel sau execute.
> - **Raw thua:** sửa JSON không chạy được; MCP sửa cell, execute, thấy ngay 4/6 bị nuốt.

In [4]:
# EDGE 2: subject-leakage check on Protocol 2 (real assign_mmfi logic)
import numpy as np
import re
from assign_subject_splits import assign_mmfi

_MMFI_RE = re.compile(r'E\d+_S(\d+)_A\d+')

# Build minimal synthetic metadata
meta = np.array([
    {'sample_id': f'E{e}_S{s}_A{a}'}
    for e in range(1, 5) for s in range(1, 41) for a in range(1, 4)
], dtype=object)

assigned = assign_mmfi(meta, protocol=2)

by_split = {}
for m in assigned:
    mm = _MMFI_RE.match(m['sample_id'])
    if mm:
        by_split.setdefault(m['split'], set()).add(mm.group(1))

tr, va, te = by_split.get('train', set()), by_split.get('val', set()), by_split.get('test', set())
leak = bool(tr & va or tr & te or va & te)
print(f'Subjects: train={len(tr)} val={len(va)} test={len(te)}')
print(f'Subject leakage detected: {leak}')
assert not leak, 'subject leakage!'
print('PASS: Protocol 2 is subject-disjoint')

Subjects: train=28 val=4 test=8
Subject leakage detected: False
PASS: Protocol 2 is subject-disjoint


> **Nhận xét EDGE 2**
>
> - Tạo 480 samples synthetic (4 env × 40 subject × 3 action), gọi `assign_mmfi(protocol=2)` thật từ repo.
> - Kết quả: train=28, val=4, test=8 subject — khớp thiết kế Protocol 2 cross-subject.
> - `Subject leakage detected: False` → assertion pass.
> - Biến `assigned` được tạo ở đây và **sống trong kernel** — ô 4, 5 phụ thuộc nó.
> - **Raw thua:** đọc code biết logic, nhưng không verify runtime assertion được.

In [5]:
# EDGE 3: depends on `assigned` from previous cell — will break after kernel restart
proto2_counts = {}
for m in assigned:
    proto2_counts[m['split']] = proto2_counts.get(m['split'], 0) + 1
print('Split counts from prior cell state:', proto2_counts)

Split counts from prior cell state: {'train': 336, 'test': 96, 'val': 48}


> **Nhận xét EDGE 3**
>
> - Cell này **không tạo data**, chỉ dùng biến `assigned` từ ô trên.
> - Kết quả: `train=336, test=96, val=48` → tổng 480, tỉ lệ ~70/20/10 đúng Protocol 2.
> - Sau `restart_notebook`, chạy ô này một mình → `NameError: assigned`.
> - MCP recovery: chạy lại chain **ô 2 → 3 → 4**.
> - **Raw thua:** file `.ipynb` trông ổn nhưng kernel trống — không biết cell nào cần re-run.

In [8]:
# EDGE 4: wrong subject digit extraction — causes false leakage alarm
import re

_BROKEN_RE = re.compile(r'E\d+_S(\d+)_A\d+')

by_split = {}
for m in assigned:
    mm = _BROKEN_RE.match(m['sample_id'])
    if mm:
        subj = mm.group(1)
        key = subj[-1]  # FIX: last digit, matches _mmfi_split logic
        by_split.setdefault(m['split'], set()).add(key)

tr, te = by_split.get('train', set()), by_split.get('test', set())
overlap = tr & te
print(f'Subject keys train={tr} test={te}')
print(f'Overlap (subject leakage): {overlap}')
assert not overlap, f'leakage via wrong digit: {overlap}'

Subject keys train={'2', '9', '1', '6', '4', '7', '3'} test={'0', '5'}
Overlap (subject leakage): set()


> **Nhận xét EDGE 4**
>
> - Bug tinh vi: `subj[0]` (chữ số đầu) vs `subj[-1]` (chữ số cuối). Ví dụ `S05` → `'0'` (train) thay vì `'5'` (test).
> - Trước fix: overlap `{'2','4','1','3'}` → `AssertionError`.
> - Sau fix: train keys `{'2','9','1','6','4','7','3'}`, test keys `{'0','5'}`, overlap = `set()`.
> - `execution_count: 8` — cell chạy nhiều nhất, phản ánh debug loop fail → fix → verify.
> - **Raw thua:** sửa JSON không thấy overlap set; MCP thấy traceback + sửa 1 dòng.

## Edge case results

| Case | Raw approach | MCP approach |
|------|--------------|--------------|
| Silent fallback on bad IDs | Miss unless you run code | Caught in EDGE 1 output |
| Kernel restart | Notebook looks fine, cells fail mysteriously | NameError + re-run chain 2->3->4 |
| Wrong digit bug | Hard to spot in static read | AssertionError with overlap set, surgical fix |
| Dependency chain | Manual guess which cells to re-run | Execute in order until green |

**Why raw loses:** `.ipynb` JSON has no execution state, no traceback, no kernel memory.

> **Nhận xét tổng thể**
>
> - Notebook demo **runtime truth**, không chỉ "chạy code được".
> - Execution count tăng dần (3 → 4 → 5 → 8) = agent re-run có hệ thống sau lỗi.
> - Output là bằng chứng: overlap set, split counts, silent fallback `'train'`.
> - Code import trực tiếp từ `scripts/assign_subject_splits.py` — không mock.
> - **Kết luận:** MCP thắng raw ở mọi edge case có phụ thuộc execution state.